In [2]:
#!/usr/bin/env python
# coding: utf-8

# # FusedHexapodModel V4.0 — Phase 1 Full Training
# 1 shared backbone: MNV2-1.4
# 4 Heads: Stereo + YOLO (35 classes) + Seg (6 depth-derived classes) + Surface Normals
# Single-channel grayscale input. No teacher dependencies for Seg.

# In[1]:


from albumentations.pytorch import ToTensorV2
from contextlib import nullcontext
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import PowerNorm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from scipy.ndimage import sobel as scipy_sobel, uniform_filter
import torch, torchvision, timm
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2, os, glob, re, math, time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# =====================================================================
# V4 CONFIG
# =====================================================================
CONFIG = {
    'img_height': 480, 'img_width': 640,
    'num_det_classes': 40,   # ✅ V2.9: 40 robot-relevant classes   
    'num_seg_classes': 6,    # ✅ V2.9: depth-derived geometric classes (was 11)
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 24,
    'batch_size': 8,           #formerly 4 
    'ACCUMULATION_STEPS': 12,  #not touched
    'lr': 2e-4,                # Reduced: 4e-4 caused inf gradients at step 0
    'start_epoch': 0,         # Offset für TensorBoard und Scheduler
    'num_epochs': 25,
    'normals_to_stereo_epoch': 5,  # <--- NEU: Ab dieser Epoche bekommt Stereo die Normalen!
    'num_workers': 3,
    'save_dir': "./checkpoints",
    'PHASE2_EPOCH': 25,     # Phase 2 starts after Phase 1
    # TartanAir camera params (after resize to 640x480)
    'tartan_fx': 320.0,
    'tartan_fy': 320.0,      # ✅ FIXED: Depth is natively 480x640, no resize → fy=fx=320
    'tartan_baseline': 0.25,
    # Seg class weights (inverse frequency, tune after first epoch)
    'seg_class_weights': [1.0, 3.0, 1.0, 2.0, 2.0, 0.5],  # WALL 0.5× (suppress), STEP 10×, OBSTACLE/VEG 5×
    'VIS_THRESH': 0.30,
    'start_step': 0,
}

SEG_CLASS_NAMES = ['WALKABLE', 'STEP', 'WALL', 'OBSTACLE', 'NAV_ANCHOR', 'VOID']


# 40 Robot-Relevant COCO Classes
ROBOT_CAT_IDS = [1,2,3,4,6,8,10,11,13,14,15,16,17,18,27,28,31,33,44,47,51,
                  62,63,64,65,67,70,72,73,75,76,77,78,79,81,82,84,85,86,88]
robot_cat_to_continuous = {cid: idx for idx, cid in enumerate(ROBOT_CAT_IDS)}

print(f"V2.10 Config: {CONFIG['num_det_classes']} det classes, {CONFIG['num_seg_classes']} seg classes")
print(f"Seg classes: {SEG_CLASS_NAMES}")

Device: cuda
  GPU: NVIDIA GeForce RTX 3080 Ti
  VRAM: 12.0 GB
V2.10 Config: 40 det classes, 6 seg classes
Seg classes: ['WALKABLE', 'STEP', 'WALL', 'OBSTACLE', 'NAV_ANCHOR', 'VOID']


## Cell 2: V4.0 Model Architecture with deployment adaptations
Paste the **complete** Cell 3 from `FusedHexapodModel_V4_0-Phase-1.ipynb` here.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import math
# Update Version 4.0: Introduce helper to replace AvgPool2D for Hailo performance optimization
class LearnablePool(nn.Module):
    """Ersetzt AvgPool durch eine Conv, die der NPU schneller verarbeitet."""
    def __init__(self, ch, kernel_size, stride=None):
        super().__init__()
        stride = stride or kernel_size
        self.pool = nn.Conv2d(ch, ch, kernel_size, stride=stride,
                              groups=ch, bias=False)  # Depthwise!
        # Initialisiere als gleichmäßige Mittelung
        with torch.no_grad():
            self.pool.weight.fill_(1.0 / (kernel_size * kernel_size 
                if isinstance(kernel_size, int) else kernel_size[0]*kernel_size[1]))
    def forward(self, x):
        return self.pool(x)

# --- Hailo-8 Compatible Building Blocks (unchanged from V2.5) ---
class DWSepConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, bias=True, dilation=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride=stride, padding=padding,
                            dilation=dilation, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
    def forward(self, x):
        return self.pw(self.dw(x))

# --- FPN Neck (unchanged from V2.5) ---
FPN_CH = 64

class LightFPNNeck(nn.Module):
    def __init__(self, ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH):
        super().__init__()
        self.lat_s8  = nn.Conv2d(ch_s8,  fpn_ch, 1, bias=False)
        self.lat_s16 = nn.Conv2d(ch_s16, fpn_ch, 1, bias=False)
        self.lat_s32 = nn.Conv2d(ch_s32, fpn_ch, 1, bias=False)
        
        # 🚨 PATCH V3.0: RepConv statt DWSepConv für kräftigere Features!
        self.smooth_s8  = RepConv(fpn_ch, fpn_ch)
        self.smooth_s16 = RepConv(fpn_ch, fpn_ch)
        self.bu_s16 = RepConv(fpn_ch, fpn_ch, stride=2)
        self.bu_s32 = RepConv(fpn_ch, fpn_ch, stride=2)

    def forward(self, f_s8, f_s16, f_s32):
        p32 = self.lat_s32(f_s32)
        p16 = self.lat_s16(f_s16) + F.interpolate(p32, scale_factor=2, mode='nearest')
        p8  = self.lat_s8(f_s8) + F.interpolate(p16, scale_factor=2, mode='nearest')
        p8  = self.smooth_s8(p8)
        p16 = self.smooth_s16(p16) + self.bu_s16(p8)
        p32 = p32 + self.bu_s32(p16)
        return p8, p16, p32

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2  # Hidden Channels
        self.cv1 = nn.Sequential(nn.Conv2d(c1, c_, 1, 1, bias=False), nn.BatchNorm2d(c_), nn.ReLU6(inplace=True))
        self.cv2 = nn.Sequential(nn.Conv2d(c_ * 4, c2, 1, 1, bias=False), nn.BatchNorm2d(c2), nn.ReLU6(inplace=True))
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        # Cat von Original + 3 MaxPool-Stufen
        return self.cv2(torch.cat((x, y1, y2, y3), 1))
# Version 4.0 Update: Remove CBAM and related classes (ChannelAttention & SpatialAttention)
'''
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1   = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(x_cat))

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x
'''

class GeometryStem(nn.Module):
    def __init__(self, in_ch=32, out_ch=32): # MobileNetV3 s4 hat oft 24 ch
        super().__init__()
        # Zwei Schichten für mehr Reife in den Features
        self.stem = nn.Sequential(
            RepConv(in_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=False),
            RepConv(out_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=False)
        )

    def forward(self, x):
        return self.stem(x)

import torch
import torch.nn as nn

class AddCoords(nn.Module):
    def __init__(self, h, w, deploy=False):
        super().__init__()
        self.deploy = deploy
        y = torch.linspace(-1, 1, h).view(1, 1, h, 1).expand(1, 1, h, w).contiguous().clone()
        x = torch.linspace(-1, 1, w).view(1, 1, 1, w).expand(1, 1, h, w).contiguous().clone()
        self.register_buffer('y_coords', y)
        self.register_buffer('x_coords', x)

    def forward(self, x_in):
        if self.deploy:
            return torch.cat([x_in, self.y_coords, self.x_coords], dim=1)
        else:
            b = x_in.shape[0]
            return torch.cat([
                x_in,
                self.y_coords.expand(b, -1, -1, -1),
                self.x_coords.expand(b, -1, -1, -1)
            ], dim=1)

class CoordConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, h, w, deploy=False, kernel_size=3, stride=1, padding=1, bias=False):
        super().__init__()
        self.add_coords = AddCoords(h, w, deploy=deploy)
        self.conv = nn.Conv2d(in_channels + 2, out_channels, kernel_size=kernel_size, 
                              stride=stride, padding=padding, bias=bias)
        
    def forward(self, x):
        return self.conv(self.add_coords(x))
    
import torch
import torch.nn as nn
import torch.nn.functional as F

class RepConv(nn.Module):
    """
    Re-Parameterized Convolution:
    Training: 3x3 Conv + 1x1 Conv + Identity (Parallel)
    Inferenz: Eine einzige 3x3 Conv (zusammengefaltet)
    """
    def __init__(self, c1, c2, kernel_size=3, stride=1, padding=1, deploy=False):
        super().__init__()
        self.deploy = deploy
        self.c1 = c1
        self.c2 = c2
        self.stride = stride
        self.padding = padding
        self.act = nn.ReLU(inplace=True)

        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=True)
        else:
            # Identity Branch (nur möglich wenn Dimensionen gleich bleiben)
            self.rbr_identity = nn.BatchNorm2d(c1) if c2 == c1 and stride == 1 else None
            # 3x3 Branch
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False),
                nn.BatchNorm2d(c2)
            )
            # 1x1 Branch
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(c1, c2, 1, stride, 0, bias=False),
                nn.BatchNorm2d(c2)
            )

    def forward(self, x):
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        
        id_out = 0 if self.rbr_identity is None else self.rbr_identity(x)
        return self.act(self.rbr_dense(x) + self.rbr_1x1(x) + id_out)

    def get_equivalent_kernel_bias(self):
        # 3x3 Kernel extrahieren
        kernel3x3, bias3x3 = self._fuse_bn_tensor(self.rbr_dense)
        # 1x1 Kernel extrahieren und auf 3x3 padden
        kernel1x1, bias1x1 = self._fuse_bn_tensor(self.rbr_1x1)
        kernel1x1 = F.pad(kernel1x1, [1, 1, 1, 1])
        # Identity Kernel erstellen (nur 1en in der Mitte)
        kernelid, biasid = self._fuse_bn_tensor(self.rbr_identity)
        
        return kernel3x3 + kernel1x1 + kernelid, bias3x3 + bias1x1 + biasid

    def _fuse_bn_tensor(self, branch):
        if branch is None:
            return torch.zeros((self.c2, self.c1, 3, 3), device=self.rbr_dense[0].weight.device), torch.zeros(self.c2, device=self.rbr_dense[0].weight.device)
        if isinstance(branch, nn.BatchNorm2d):
            # Trick: Fake-Kernel für Identity
            kernel = torch.zeros((self.c1, self.c1, 3, 3), device=branch.weight.device)
            for i in range(self.c1): kernel[i, i, 1, 1] = 1.0
            return self._fuse_bn(kernel, branch.running_mean, branch.running_var, branch.weight, branch.bias, branch.eps)
        else:
            return self._fuse_bn(branch[0].weight, branch[1].running_mean, branch[1].running_var, branch[1].weight, branch[1].bias, branch[1].eps)

    def _fuse_bn(self, kernel, mean, var, gamma, beta, eps):
        std = (var + eps).sqrt()
        t = (gamma / std).reshape(-1, 1, 1, 1)
        return kernel * t, beta - mean * gamma / std
        
    def switch_to_deploy(self):
        if self.deploy: return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(self.c1, self.c2, 3, self.stride, self.padding, bias=True)
        self.rbr_reparam.weight.data = kernel
        self.rbr_reparam.bias.data = bias
        # Lösche Trainings-Branches, um RAM zu befreien!
        for attr in ['rbr_dense', 'rbr_1x1', 'rbr_identity']:
            if hasattr(self, attr): delattr(self, attr)
        self.deploy = True
    
# --- Cost Volume (Korrigiert: Universelle Metrik) ---
class CoarseCostVolume(nn.Module):
    def __init__(self, max_disp, in_channels, deploy=False):
        super().__init__()
        self.max_disp = max_disp
        self.deploy = deploy
        self.corr = nn.Conv2d(in_channels * 2, 1, 1, bias=True)
        if self.deploy:
            self.corr_grouped = nn.Conv2d(max_disp * in_channels * 2, max_disp, 1, groups=max_disp)
        
    def forward(self, feat_l, feat_r):
        if self.deploy:
            all_shifted = []
            for d in range(self.max_disp):
                if d == 0:
                    shifted = feat_r
                else:
                    shifted = F.pad(feat_r, (d, 0, 0, 0))[:, :, :, :-d]
                all_shifted.append(torch.cat([feat_l, shifted], dim=1))
            
            # 🚀 KANAL-STACK statt BATCH-STACK
            # Shape: [1, 24 * (C*2), 60, 80] -> z.B. [1, 1536, 60, 80]
            x = torch.cat(all_shifted, dim=1)
            
            # Die Grouped Conv rechnet alle 24 Blöcke strikt getrennt in einem Rutsch
            x = self.corr_grouped(x) # Shape: [1, 24, 60, 80]
            return x
            
        else:
            # 🧠 TRAINING (GPU)
            B, C, H, W = feat_l.shape # Hier ist das Auslesen von B völlig okay!
            cost_slices = []
            for d in range(self.max_disp):
                if d == 0:
                    cost_slices.append(torch.cat([feat_l, feat_r], dim=1))
                else:
                    shifted = torch.zeros_like(feat_r)
                    shifted[:, :, :, d:] = feat_r[:, :, :, :-d]
                    cost_slices.append(torch.cat([feat_l, shifted], dim=1))
            
            cost = torch.stack(cost_slices, dim=2)
            B, C2, D, H, W = cost.shape
            cost = cost.permute(0, 2, 1, 3, 4).reshape(B * D, C2, H, W)
            out = self.corr(cost) 
            return out.view(B, D, H, W)

# --- Refinement Stage with Edge Guidance (from V2.5 Phase 2) ---
class RefinementStage(nn.Module):
    def __init__(self, guidance_channels, scale_factor, use_edge_guidance=False, deploy=False):
        super().__init__()
        self.deploy = deploy
        self.scale_factor = scale_factor
        self.use_edge_guidance = use_edge_guidance
        extra = 1 if use_edge_guidance else 0
        self.net = nn.Sequential(
            nn.Conv2d(1 + guidance_channels + extra, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1)
        )
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def _edge_map(self, img):
        gx = F.conv2d(img, self.kx, padding=1)
        gy = F.conv2d(img, self.ky, padding=1)
        return torch.abs(gx) + torch.abs(gy)       # <-- NEU (L1-Trick)

    # In class RefinementStage(nn.Module):
    def forward(self, disparity_low, guidance, max_disp, gray_img=None): # <--- NEU: max_disp hinzugefügt
        mode = 'nearest' if self.deploy else 'bilinear' # Dynamisch umschalten
        disparity_up = F.interpolate(
            disparity_low, scale_factor=self.scale_factor,
            mode=mode, 
            align_corners=False if mode == 'bilinear' else None
        ) * self.scale_factor
        
        # ✅ PHYSIKALISCH FUNDIERTE NORMALISIERUNG
        # Wir bringen die Disparität für das CNN auf einen Prozentwert (0.0 bis 1.0)
        norm_disp = disparity_up / max_disp
        
        # Das CNN kriegt jetzt die normierte Disparität + die Guidance-Features
        inp = [norm_disp, guidance]
        
        if self.use_edge_guidance:
            assert gray_img is not None
            if gray_img.shape[-2:] != disparity_up.shape[-2:]:
                # Im Deployment 'nearest' nutzen
                if self.deploy:
                    gray_img = F.interpolate(
                        gray_img, 
                        size=disparity_up.shape[-2:],
                        mode='nearest',
                        align_corners= None
                    )
                else:
                    gray_img = F.interpolate(
                        gray_img, 
                        size=disparity_up.shape[-2:],
                        mode='bilinear',
                        align_corners= False
                    )
                
            inp.append(self._edge_map(gray_img))
            
        # Das berechnete Detail-Residual wird zur ORIGINALEN (unskalierten) Disparität addiert!
        return F.relu(disparity_up + self.net(torch.cat(inp, dim=1)))

# --- Stereo Head with Context Network ---
class HierarchicalStereoHead(nn.Module):
    def __init__(self, ch_s8, ch_s4, max_disp_s8, use_normals=True, deploy=False):
        super().__init__()
        self.max_disp_s8 = max_disp_s8
        self.use_normals = use_normals
        self.deploy = deploy
        # ====================================================================
        # 🚨 PATCH V3.0: CoordConv für absolutes räumliches Bewusstsein!
        # kernel_size=1, padding=0 sorgt dafür, dass deine Architektur 
        # exakt gleich bleibt, nur dass X/Y elegant miteingemischt werden.
        # ====================================================================
        
        # 🚨 V3.1 GEOMETRY UPGRADE: 
        # 🚨 V4.0 GEOMETRY Optimization for Hailo deployment: Reduce cost volume channels 32 -> 16 
        # Wir fügen geo_features (32 ch) hinzu. 
        # Für s8 müssen wir sie erst poolen, für s4 passen sie direkt.
        self.reduce_s8 = CoordConv2d(ch_s8 + 32, 16, h=60, w=80, deploy=deploy, kernel_size=1, padding=0, bias=False)


        # S4 Guidance: Normalen (3) + Geo (32) + Backbone (ch_s4)
        s4_guidance_ch = ch_s4 + 32 + (3 if use_normals else 0)
        self.reduce_s4 = CoordConv2d(s4_guidance_ch, 16, h=120, w=160, deploy=deploy, kernel_size=1, padding=0, bias=False)
      
        self.stereo_coarse = CoarseCostVolume(max_disp=self.max_disp_s8, in_channels=16, deploy=deploy)
        self.stereo_refine_s4 = RefinementStage(guidance_channels=16, scale_factor=2.0, deploy=deploy)
        # Update 4.0_ Remove S1 refiner
        #self.stereo_refine_s1 = RefinementStage(guidance_channels=1, scale_factor=4.0, use_edge_guidance=True)
        
        self.register_buffer('disp_reg', torch.arange(self.max_disp_s8, dtype=torch.float32).view(1, -1, 1, 1))
        self.temperature = 0.7
        self.context_weight = 0.8
        
        # ====================================================================
        # 🚨 PATCH V3.0: Refactoring des Context-Blocks (Fix 2)
        # Sauberer Code, nutzt DWSepConv für die teure mittlere Schicht!
        # ====================================================================
        self.context = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),        # 1 auf 16: normale Conv
            nn.ReLU(inplace=True),
            DWSepConv(16, 16, 3, padding=1),       # 16 auf 16: schlankes DWSepConv!
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 3, padding=1)         # 16 auf 1: Output
        )
        # 🚀 Grouped COntext für Deployment
        # Wird später von prep_stereo_for_deploy() mit Gewichten gefüllt.
        if self.deploy:
            self.context_grouped = nn.Sequential(
                nn.Conv2d(self.max_disp_s8, self.max_disp_s8 * 16, 3, padding=1, groups=self.max_disp_s8),
                nn.ReLU(inplace=True),
                nn.Conv2d(self.max_disp_s8 * 16, self.max_disp_s8 * 16, 3, padding=1, groups=self.max_disp_s8 * 16, bias=False),
                nn.Conv2d(self.max_disp_s8 * 16, self.max_disp_s8 * 16, 1, groups=self.max_disp_s8),
                nn.ReLU(inplace=True),
                nn.Conv2d(self.max_disp_s8 * 16, self.max_disp_s8, 3, padding=1, groups=self.max_disp_s8)
            )
    
        # Lernbare Parameter für GeometryStem
        self.geo_downsample = nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1, bias=False)

    # 🚨 V3.1: Signatur um geo_features_l und geo_features_r erweitert
    def forward(self, l_s8, r_s8, l_s4, l_img_raw, normals_s4=None, geo_features_l=None, geo_features_r=None):
        
        # --- SICHERHEITSNETZ (Für TensorBoard Tracer / Alte Checkpoints) ---
        if geo_features_l is None:
            B, _, H_s4, W_s4 = l_s4.shape
            geo_features_l = torch.zeros(B, 32, H_s4, W_s4, device=l_s4.device, dtype=l_s4.dtype)
        if geo_features_r is None:
            B, _, H_s4, W_s4 = l_s4.shape
            geo_features_r = torch.zeros(B, 32, H_s4, W_s4, device=l_s4.device, dtype=l_s4.dtype)

        # --- STUFE 1: S8 Coarse Matching ---
        # Da geo_features auf s4 (z.B. 160x120) sind, poolen wir sie für s8 (80x60)
        #geo_l_s8 = F.avg_pool2d(geo_features_l, kernel_size=2, stride=2)
        #geo_r_s8 = F.avg_pool2d(geo_features_r, kernel_size=2, stride=2)
        
        # geo_downsample statt avg_pool2d, um die Kantenschärfe zu erhalten
        geo_l_s8 = self.geo_downsample(geo_features_l)
        geo_r_s8 = self.geo_downsample(geo_features_r)
        
        # Jetzt mit Backbone-Features mischen (+ 32 Kanäle!)
        feat_l_s8 = self.reduce_s8(torch.cat([l_s8, geo_l_s8], dim=1))
        feat_r_s8 = self.reduce_s8(torch.cat([r_s8, geo_r_s8], dim=1))
        
        # --- STUFE 2: S4 Refinement Guidance ---
        if self.use_normals:
            if normals_s4 is not None:
                norm_in = normals_s4
            else:
                norm_in = torch.zeros(l_s4.size(0), 3, l_s4.size(2), l_s4.size(3), device=l_s4.device, dtype=l_s4.dtype)
            
            # Alle drei Quellen: Backbone (s4) + GeoStem (s4) + Normals (s4)
            l_s4_combined = torch.cat([l_s4, geo_features_l, norm_in], dim=1)
        else:
            l_s4_combined = torch.cat([l_s4, geo_features_l], dim=1)
            
        feat_l_s4 = self.reduce_s4(l_s4_combined)
        
        # --- STUFE 3: Stereo Prozess ---
        vol_s8 = self.stereo_coarse(feat_l_s8, feat_r_s8)
        
        # Kontext-Netzwerk
        if self.deploy:
            # Das gesamte Context-Netz wird als eine große GroupedConv ausgeführt!
            vol_ctx = self.context_grouped(vol_s8)
            #print("deploy")
            #print("vol_s8 shape:", vol_s8.shape)
            #print("vol_ctx shape:", vol_ctx.shape)
            #print("vol_s8 max:", vol_s8.abs().max().item())
            #print("vol_ctx max:", vol_ctx.abs().max().item())
            
        else:
            # 🧠 TRAINING FÜR GPU (mit Batch-Logik)
            B, D, H, W = vol_s8.shape
            vol_reshaped = vol_s8.view(B * D, 1, H, W)
            vol_ctx = self.context(vol_reshaped).view(B, D, H, W)
            #print("training")
            #print("vol_s8 shape:", vol_s8.shape)
            #print("vol_ctx shape:", vol_ctx.shape)
            #print("vol_s8 max:", vol_s8.abs().max().item())
            #print("vol_ctx max:", vol_ctx.abs().max().item())
        
        vol_s8 = self.context_weight * vol_ctx + (1.0 - self.context_weight) * vol_s8


        # ====================================================================
        # 🚀 AUFGELÖSTE SOFTMAX-BERECHNUNG FÜR HAILO
        # ====================================================================
        if self.deploy:
            # 1. Skalierung mit Temperature
            vol_scaled = vol_s8 / self.temperature
            # 2. Maximum für numerische Stabilität (INT8 Overflow verhindern)
            v_max, _ = torch.max(vol_scaled, dim=1, keepdim=True)
            # 3. Exponentialfunktion
            v_exp = torch.exp(vol_scaled - v_max)
            # 4. Summe + Anti-Fusion Trick (1e-6 verhindert, dass ONNX es wieder zu Softmax macht)
            v_sum = torch.sum(v_exp, dim=1, keepdim=True) + 1e-6
            # 5. Wahrscheinlichkeit
            prob_s8 = v_exp / v_sum
            disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)
            #print("prob deploy:", (prob_s8).abs().max().item())
            #print("disp_s8 deploy:", (disp_s8).abs().max().item())
        else:
            # 🧠 TRAINING: PyTorch Standard Softmax
            prob_s8 = F.softmax(vol_s8 / self.temperature, dim=1)
            disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)
            #print("prob train:", (prob_s8).abs().max().item())
            #print("disp_s8 train:", (disp_s8).abs().max().item())
        
        # ✅ V4.0: Confidence Gate — NUR bei Inferenz, nicht beim Training
        if not self.training:
            confidence = prob_s8.max(dim=1, keepdim=True)[0]
            gate = torch.clamp((confidence - 0.10) / 0.10, 0.0, 1.0)
            disp_s8 = disp_s8 * gate


        max_disp_s4 = self.max_disp_s8 * 2.0
        disp_s4 = self.stereo_refine_s4(disp_s8, feat_l_s4, max_disp=max_disp_s4)
        
        # max_disp_s1 = max_disp_s4 * 4.0
        # final_disp = self.stereo_refine_s1(disp_s4, l_img_raw, max_disp=max_disp_s1, gray_img=l_img_raw)
        #print("disp_s4 max:", disp_s4.abs().max().item())
        #print("final_disp max:", final_disp.abs().max().item())
        return disp_s4, disp_s8

# ✅ V3.0 Normals Head - Multi-Scale Fusion (s4 + s8), Up-Sampling, Image-Guided Refinement, CoordConv
class NormalsHead(nn.Module):
    def __init__(self, ch_s4, ch_s8, deploy=False): # NEU: deploy flag
        super().__init__()
        self.deploy = deploy
        # Stage 1: Multi-Scale Fusion (s4 + s8)
        self.s8_adapt = nn.Conv2d(ch_s8, 32, kernel_size=1)
        
        # 🚨 V3.1 FIX: Backbone (ch_s4) + s8_adapt (32) + geo_features (32)
        fused_ch = ch_s4 + 32 + 32 
        
        # ====================================================================
        # 🚨 PATCH V3.0: CoordConv für den NormalsHead!
        # ====================================================================
        self.stage1 = nn.Sequential(
            CoordConv2d(fused_ch, 96, h=120, w=160, deploy=deploy, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.Conv2d(96, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(inplace=True)
        )
        self.coarse_out = nn.Conv2d(32, 3, 3, padding=1)

        # Stage 3: Image-Guided Refinement
        # Inputs: 3 (Coarse Normals) + 1 (Gray Img) + 1 (Sobel Edges) = 5
        # Update 4.0: Remove high-res refiners to save Hailo resources
        '''
        self.refiner = nn.Sequential(
            nn.Conv2d(5, 32, 3, padding=1),
            
            # ====================================================================
            # 🚨 PATCH V3.0: Clean Code mit DWSepConv! 
            # (Ersetzt die 4 manuellen Layer von vorher)
            # ====================================================================
            DWSepConv(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            
            DWSepConv(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(32, 3, 3, padding=1)
        )
        
        # Fest verdrahtete Sobel-Filter (keine trainierbaren Parameter)
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def get_edges(self, img):
        gx = F.conv2d(img, self.sobel_x, padding=1)
        gy = F.conv2d(img, self.sobel_y, padding=1)
        # 🚨 100% Hailo-Safe: L1-Norm (Manhattan-Distanz) statt Wurzel
        # Das löst auch das FP16 NaN-Problem automatisch, da es keine Wurzel mehr gibt!
        return torch.abs(gx) + torch.abs(gy)
    
    '''

    def forward(self, l_s4, l_s8, gray_img, geo_features=None):
        s8_adapted = self.s8_adapt(l_s8)
        #s8_adapted = F.interpolate(s8_adapted, size=l_s4.shape[2:], 
        #                           mode='bilinear', align_corners=False)
        s8_adapted = F.interpolate(s8_adapted, size=l_s4.shape[2:], 
                                   mode='nearest', align_corners=None)
        
        if geo_features is not None:
            l_s4_combined = torch.cat([l_s4, s8_adapted, geo_features], dim=1)
        else:
            dummy_geo = torch.zeros(l_s4.size(0), 32, l_s4.size(2), l_s4.size(3),
                                    device=l_s4.device, dtype=l_s4.dtype)
            l_s4_combined = torch.cat([l_s4, s8_adapted, dummy_geo], dim=1)
        
        feat_s4 = self.stage1(l_s4_combined)
        coarse_normals_s4 = self.coarse_out(feat_s4)
        
        # L2-Normalisierung direkt auf s4 (Training + Deploy identisch!)
        if self.deploy:
            v_max, _ = torch.max(torch.abs(coarse_normals_s4), dim=1, keepdim=True)
            scaled = coarse_normals_s4 / (v_max + 1e-6)
            l2 = torch.sqrt(torch.sum(scaled * scaled, dim=1, keepdim=True))
            divisor = torch.clamp(l2 * (v_max + 1e-6), min=1e-4)
            normals_s4 = coarse_normals_s4 / divisor
        else:
            normals_s4 = F.normalize(coarse_normals_s4, p=2, dim=1, eps=1e-4)
        
        # Nur s4 zurückgeben — kein s1 mehr!
        return normals_s4

# ✅ V3.1: LRASPPHead with Normals input + 6 classes
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes, normals_ch=3):
        super().__init__()
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        # Update Version 4.0: Replace AvgPool2d with LernablePool()
        self.scale_high = nn.Sequential(
            LearnablePool(high_ch, kernel_size=(30, 40)),  # statt nn.AvgPool2d
            nn.Conv2d(high_ch, 128, 1, bias=False),
            nn.Sigmoid()
        )
        # ✅ V2.9: low_classifier takes backbone features + predicted normals
        self.low_classifier = nn.Conv2d(low_ch + normals_ch, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)
        # Dilated conv also gets normals (activated in Phase 2)
        self.mid_classifier = nn.Conv2d(128 + normals_ch, num_classes, 3, padding=2, dilation=2)
        self.use_mid = True
        
    def forward(self, x_low, x_high, normals_s4=None):
        out = self.cbr_high(x_high) * self.scale_high(x_high)
        #out = F.interpolate(out, scale_factor=4.0, mode='bilinear', align_corners=False)
        out = F.interpolate(out, scale_factor=4.0, mode='nearest', align_corners=None)
        
        if normals_s4 is not None:
            low_in = torch.cat([x_low, normals_s4], dim=1)
        else:
            # Make it Hailo compatible (kein F.pad auf Channels):
            dummy_normals = torch.zeros(x_low.size(0), 3, x_low.size(2), x_low.size(3), device=x_low.device)
            low_in = torch.cat([x_low, dummy_normals], dim=1)
            
        result = self.low_classifier(low_in) + self.high_classifier(out)
        
        if self.use_mid:
            if normals_s4 is not None:
                mid_in = torch.cat([out, normals_s4], dim=1)
            else:
                # ✅ KORRIGIERT: Auch hier dummy_normals mit torch.cat statt F.pad
                dummy_normals_mid = torch.zeros(out.size(0), 3, out.size(2), out.size(3), device=out.device)
                mid_in = torch.cat([out, dummy_normals_mid], dim=1)
                
            result = result + self.mid_classifier(mid_in)   
        return result
        
# --- YOLO Heads (unchanged structure, 40 classes) ---
class DecoupledHead(nn.Module):
    def __init__(self, ch_in, num_classes, h, w, deploy=False, width=128):
        super().__init__()
        
        self.coord_conv_cls = CoordConv2d(ch_in, width, h=h, w=w, deploy=deploy, kernel_size=3, padding=1)
        self.coord_conv_reg = CoordConv2d(ch_in, width, h=h, w=w, deploy=deploy, kernel_size=3, padding=1)
        
        # 🚨 PATCH V3.0: RepConv integriert die ReLU bereits intern!
        self.cls_convs = nn.Sequential(
            self.coord_conv_cls,
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            RepConv(width, width) # ⬅️ RepConv statt DWSepConv
        )
        
        self.reg_convs = nn.Sequential(
            self.coord_conv_reg,
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            RepConv(width, width) # ⬅️ RepConv statt DWSepConv
        )
        
        self.cls_pred = nn.Conv2d(width, num_classes, 1)
        self.reg_pred = nn.Conv2d(width, 4, 1)
        self.obj_pred = nn.Conv2d(width, 1, 1)
    def forward(self, x):
        cls_feat = self.cls_convs(x); reg_feat = self.reg_convs(x)
        return torch.cat([self.reg_pred(reg_feat), self.obj_pred(reg_feat), self.cls_pred(cls_feat)], dim=1)

class YOLOHead(nn.Module):
    def __init__(self, fpn_ch=FPN_CH, num_classes=40, deploy=False):
        super().__init__()
        self.head_s8  = DecoupledHead(fpn_ch, num_classes, h=60, w=80, deploy=deploy, width=128)
        self.head_s16 = DecoupledHead(fpn_ch, num_classes, h=30, w=40, deploy=deploy, width=128)
        self.head_s32 = DecoupledHead(fpn_ch, num_classes, h=15, w=20, deploy=deploy, width=128)
    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]

# =====================================================================
# ✅ V4.0: FusedHexapodModel — 1-Channel Input, 5 Outputs
# =====================================================================
class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.deploy_mode = config.get('deploy', False)
        self.backbone = timm.create_model('mobilenetv2_140.ra_in1k', pretrained=True,
                                   features_only=True, out_indices=(1, 2, 3, 4))
        feat_info = self.backbone.feature_info.channels()
        ch_s4, ch_s8, ch_s16, ch_s32 = feat_info  # 24, 32, 96, 320

        # ✅ V2.9: Patch first conv to 1-channel input
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                              stride=old_conv.stride, padding=old_conv.padding, bias=False)
        # Merge RGB weights via luminance formula
        with torch.no_grad():
            w = old_conv.weight.data  # [C_out, 3, kH, kW]
            new_conv.weight.data = w[:, 0:1]*0.299 + w[:, 1:2]*0.587 + w[:, 2:3]*0.114
        self.backbone.conv_stem = new_conv
        print(f"  ✅ Backbone first conv: 3→1 channel (luminance merge)")
        
        # 🚨 PATCH V3.0: SPPF initialisieren (nimmt ch_s32 dynamisch von timm!)
        self.sppf = SPPF(c1=ch_s32, c2=ch_s32, k=5)
        print(f"  ✅ SPPF Module injected at s32 ({ch_s32} channels)")
        
        disp_steps = config.get('internal_disp_steps', 48)
        self.stereo_head = HierarchicalStereoHead(ch_s8, ch_s4, max_disp_s8=disp_steps, deploy=self.deploy_mode)
        self.normals_head = NormalsHead(ch_s4, ch_s8, deploy=self.deploy_mode)
        self.seg_head = LRASPPHead(ch_s4, ch_s16, config['num_seg_classes'], normals_ch=3)
        self.fpn_neck = LightFPNNeck(ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH)

        # 🚨 PATCH V4.0: Remove CBAM Attention für die FPN-Outputs
        # self.cbam_s8  = CBAM(FPN_CH)
        # self.cbam_s16 = CBAM(FPN_CH)
        # self.cbam_s32 = CBAM(FPN_CH)
        # print(f"  ✅ CBAM Attention Modules injected after FPN")

        self.geo_stem = GeometryStem(in_ch=ch_s4, out_ch=32)
        print(f"  ✅ Geometry Stem Modules injected after Backbone")

        self.yolo_head = YOLOHead(fpn_ch=FPN_CH, num_classes=config['num_det_classes'], deploy=self.deploy_mode)

        total_params = sum(p.numel() for p in self.parameters())
        print(f"  Total parameters: {total_params:,}")
        print(f"  Normals Head: {sum(p.numel() for p in self.normals_head.parameters()):,}")
        print(f"  Seg Head: {sum(p.numel() for p in self.seg_head.parameters()):,}")
        print(f"  YOLO Head: {sum(p.numel() for p in self.yolo_head.parameters()):,}")

    def forward(self, x_left, x_right, use_normals_for_stereo=False):
        # Backbone & FPN
        features_l = self.backbone(x_left)
        # ====================================================================
        # 🚨 NEU V3.1: GEOMETRY STEM (High-Res Pfad)
        # Nutzt features_l[0] (s4 / 160x120), um scharfe Kanten-Features zu extrahieren
        # ====================================================================
        geo_feat_l = self.geo_stem(features_l[0])
        # Beide Auflösungen vom Normals-Head abgreifen (Neu: mit s8 und gray_img)
        # Normals bekommt jetzt die Geo-Features zusätzlich
        normals_s4 = self.normals_head(
            features_l[0], 
            features_l[1], 
            x_left, 
            geo_features=geo_feat_l # 👈 NEU: High-Res Support
        )
        
        disp_s4, disp_s8 = None, None
        
        # ✅ FIX: Nur Stereo ausführen, wenn wir auch ein rechtes Bild haben (TartanAir)
        if x_right is not None:
            with torch.no_grad():
                features_r = self.backbone(x_right)
                # Auch für das rechte Bild brauchen wir die Geo-Features für das Matching!
                geo_feat_r = self.geo_stem(features_r[0])
            
            # 🚨 FIX: normals_s4.detach() verhindert, dass Stereo-Gradienten den NormalsHead zerstören!
            if not self.deploy_mode:
                normals_s4_for_stereo = normals_s4.detach() if (use_normals_for_stereo and normals_s4 is not None) else None
            else:
                normals_s4_for_stereo = normals_s4
            
            # Stereo Head (nutzt jetzt die entkoppelten Normalen UND das Graustufenbild)
            # Stereo Head bekommt jetzt geo_feat_l UND geo_feat_r
            disp_s4, disp_s8 = self.stereo_head(
                features_l[1], features_r[1], features_l[0], x_left,
                normals_s4=normals_s4_for_stereo,
                geo_features_l=geo_feat_l,
                geo_features_r=geo_feat_r
            )
        else:
            disp_s4, disp_s8 = None, None
            
        # Seg bekommt ebenfalls die S4-Normalen (160x120)
        if self.deploy_mode:
            normals_for_others = normals_s4
        else:
            normals_for_others = normals_s4.detach() if (use_normals_for_stereo and normals_s4 is not None) else None
        seg = self.seg_head(features_l[0], features_l[2], normals_s4=normals_for_others)

        # 🚨 PATCH V3.0: SPPF auf die tiefste Ebene anwenden
        f_s32_sppf = self.sppf(features_l[3])
        # FPN_Neck mit der gepatchten s32-Map aufrufen
        fpn_s8, fpn_s16, fpn_s32 = self.fpn_neck(features_l[1], features_l[2], f_s32_sppf)
        
        # ====================================================================
        # 🚨 PATCH V4.0: Remove CBAM Attention 
        # ====================================================================
        # fpn_s8_att  = self.cbam_s8(fpn_s8)
        # fpn_s16_att = self.cbam_s16(fpn_s16)
        # fpn_s32_att = self.cbam_s32(fpn_s32)

        # YOLO bekommt jetzt die gefilterten Attention-Features!
        det = self.yolo_head(fpn_s8, fpn_s16, fpn_s32)
        yolo_s8, yolo_s16, yolo_s32 = det
        '''
        if not self.deploy_mode:
            #print("training: normals_s4_for_stereo is None:", normals_s4_for_stereo is None)
            yolo_s8, yolo_s16, yolo_s32 = det
            return final_disp, seg, disp_s8, normals_s1, yolo_s8, yolo_s16, yolo_s32
        else:
            #print("deploy: normals_s4_for_stereo is None:", normals_s4_for_stereo is None)
            yolo_s8, yolo_s16, yolo_s32 = det
            return final_disp, seg, disp_s8, normals_s1, yolo_s8, yolo_s16, yolo_s32
        '''
        return disp_s4, seg, disp_s8, normals_s4, yolo_s8, yolo_s16, yolo_s32

model = FusedHexapodModel(CONFIG).to(DEVICE)
print(f"\n✅ V4.00 Model created on {DEVICE}")

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  ✅ Backbone first conv: 3→1 channel (luminance merge)
  ✅ SPPF Module injected at s32 (448 channels)
  ✅ Geometry Stem Modules injected after Backbone
  Total parameters: 6,094,963
  Normals Head: 160,931
  Seg Head: 206,342
  YOLO Head: 1,462,791

✅ V4.00 Model created on cuda


In [4]:
import torch
import torch.nn as nn
import os
import copy

# =====================================================================
# 🛠️ KONFIGURATION
# =====================================================================
checkpoint_paths = [
    "/home/slarc/jupyter/checkpoints/checkpoint_v4_0_best.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_56672.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_55440.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_54208.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_52976.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_51744.pth"
]

# (Angenommen CONFIG und DEVICE sind hier definiert)
# CONFIG = {...}
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def prep_stereo_for_deploy(stereo_head):
    """
    Transformiert die Trainings-Gewichte in Grouped Convolutions für den Hailo.
    Wird auf das DUMMY-Modell angewendet, um das State-Dict zu generieren!
    """
    if stereo_head.deploy: return
    stereo_head.deploy = True
    stereo_head.stereo_coarse.deploy = True
    max_disp = stereo_head.max_disp_s8

    # 1. Cost Volume Conv umwandeln
    old_corr = stereo_head.stereo_coarse.corr
    in_ch = old_corr.in_channels
    corr_grouped = nn.Conv2d(max_disp * in_ch, max_disp, 1, groups=max_disp).to(old_corr.weight.device)
    corr_grouped.weight.data = old_corr.weight.data.repeat(max_disp, 1, 1, 1)
    corr_grouped.bias.data = old_corr.bias.data.repeat(max_disp)
    stereo_head.stereo_coarse.corr_grouped = corr_grouped

    # 2. Context Netz umwandeln
    ctx = stereo_head.context
    
    w0, b0 = ctx[0].weight, ctx[0].bias
    conv1 = nn.Conv2d(max_disp, max_disp * 16, 3, padding=1, groups=max_disp).to(w0.device)
    conv1.weight.data = w0.repeat(max_disp, 1, 1, 1)
    conv1.bias.data = b0.repeat(max_disp)

    w_dw = ctx[2].dw.weight
    dw = nn.Conv2d(max_disp * 16, max_disp * 16, 3, padding=1, groups=max_disp * 16, bias=False).to(w0.device)
    dw.weight.data = w_dw.repeat(max_disp, 1, 1, 1)

    w_pw, b_pw = ctx[2].pw.weight, ctx[2].pw.bias
    pw = nn.Conv2d(max_disp * 16, max_disp * 16, 1, groups=max_disp).to(w0.device)
    pw.weight.data = w_pw.repeat(max_disp, 1, 1, 1)
    pw.bias.data = b_pw.repeat(max_disp)

    w4, b4 = ctx[4].weight, ctx[4].bias
    conv3 = nn.Conv2d(max_disp * 16, max_disp, 3, padding=1, groups=max_disp).to(w0.device)
    conv3.weight.data = w4.repeat(max_disp, 1, 1, 1)
    conv3.bias.data = b4.repeat(max_disp)

    stereo_head.context_grouped = nn.Sequential(
        conv1, nn.ReLU(inplace=True),
        dw, pw, nn.ReLU(inplace=True),
        conv3
    )

# =====================================================================
# 🚀 SWA + FOLDING LOGIK (Für das Dummy-Modell)
# =====================================================================
def create_swa_and_fold(model, filepaths):
    print(f"🔄 1. Starte SWA: Mittle {len(filepaths)} Checkpoints...")
    
    swa_state_dict = None
    num_checkpoints = len(filepaths)
    
    for path in filepaths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"❌ Checkpoint nicht gefunden: {path}")
            
        print(f"📥 Lade: {path}")
        ckpt = torch.load(path, map_location='cpu')
        model_state = ckpt.get('model_state_dict', ckpt)
        
        if swa_state_dict is None:
            swa_state_dict = {k: v.clone() for k, v in model_state.items()}
        else:
            for k in swa_state_dict.keys():
                if k in model_state:
                    swa_state_dict[k] += model_state[k]
    
    # Durchschnitt berechnen
    for k in swa_state_dict.keys():
        if swa_state_dict[k].is_floating_point():
            swa_state_dict[k].div_(num_checkpoints)
        else:
            swa_state_dict[k] = torch.div(swa_state_dict[k], num_checkpoints, rounding_mode='floor')
            
    print("✅ Mittelung abgeschlossen.")
    
    print("🏗️ 2. Lade SWA-Gewichte in Trainings-Struktur...")
    missing, unexpected = model.load_state_dict(swa_state_dict, strict=False)
    unexpected_weights = [k for k in missing if 'coords' not in k]
    if unexpected_weights:
        raise RuntimeError(f"❌ Unerwartete fehlende Gewichte: {unexpected_weights}")
    
    model.eval() # WICHTIG: BatchNorm einfrieren vor dem Falten!
    
    print("🧬 3. Rufe switch_to_deploy() auf (RepConv Faltung)...")
    folded_count = 0
    for m in model.modules():
        if hasattr(m, 'switch_to_deploy'):
            m.switch_to_deploy()
            folded_count += 1
            
    print(f"✨ {folded_count} RepConv-Layer erfolgreich gefaltet!")
    return model, swa_state_dict

# =====================================================================
# 🏁 AUSFÜHRUNG & ONNX EXPORT (Live-Switch Strategie)
# =====================================================================

print("\n--- SWA, FOLDING & GEWICHTS-TRANSFORMATION ---")
# 1. Zwingend auf False, damit die RepConv-Zweige für SWA existieren!
deploy_config = copy.deepcopy(CONFIG)
deploy_config['deploy'] = False  

print("🏗️ 1. Erstelle Modell-Architektur (Trainings-Struktur)...")
model = FusedHexapodModel(deploy_config)

# 2. Lade SWA-Gewichte und falte die RepConvs
# (Da model_orig exakt die Trainings-Struktur hat, passt SWA hier zu 100% rein!)
model, raw_swa_state_dict = create_swa_and_fold(model, checkpoint_paths)

print("🧬 3. Transformiere Stereo-Gewichte für Hailo (Grouped Convs)...")
# Erstellt die grouped_convs direkt in der Instanz dieses Modells
prep_stereo_for_deploy(model.stereo_head)

print("🔄 4. Schalte gesamtes Modell live in den Deploy-Modus...")
def enable_deploy_mode_recursively(net):
    """Durchforstet das gesamte PyTorch-Modell und knipst alle Deploy-Schalter an"""
    if hasattr(net, 'deploy_mode'):
        net.deploy_mode = True
    for m in net.modules():
        if hasattr(m, 'deploy'):
            m.deploy = True

enable_deploy_mode_recursively(model)
print("🟢 Alle Module erfolgreich auf Deploy-Modus umgeschaltet!")

# 5. Backup Speichern (Das ist jetzt dein sauberes, 100% einsatzbereites Deploy-Modell)
output_path_deploy = "checkpoints/checkpoint_v4_0_deploy.pth"
os.makedirs(os.path.dirname(output_path_deploy), exist_ok=True)
torch.save(model.state_dict(), output_path_deploy)
print(f"💾 Finales Deploy-Modell gespeichert unter: {output_path_deploy}")

# ---------------------------------------------------------
# ONNX EXPORT
# ---------------------------------------------------------
print("\n🚀 5. Starte ONNX Export...")
model = model.eval().to(DEVICE) # Sicherheitshalber Eval-Modus und GPU
dummy_l = torch.randn(1, 1, 480, 640).to(DEVICE)
dummy_r = torch.randn(1, 1, 480, 640).to(DEVICE)

onnx_path = "hexapod_v4_0_deploy.onnx"

torch.onnx.export(
    model,
    (dummy_l, dummy_r),
    onnx_path,
    input_names=['input_layer1', 'input_layer2'],
    output_names=['disp_final', 'seg', 'disp_s8', 'normals_s1', 'yolo_s8', 'yolo_s16', 'yolo_s32'],
    opset_version=13,
    do_constant_folding=True
)

print(f"🎉 Export abgeschlossen! Die Datei '{onnx_path}' ist jetzt zu 100% Hailo-Ready.")

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



--- SWA, FOLDING & GEWICHTS-TRANSFORMATION ---
🏗️ 1. Erstelle Modell-Architektur (Trainings-Struktur)...
  ✅ Backbone first conv: 3→1 channel (luminance merge)
  ✅ SPPF Module injected at s32 (448 channels)


/tmp/ipykernel_664/2457640332.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location='cpu')


  ✅ Geometry Stem Modules injected after Backbone
  Total parameters: 6,094,963
  Normals Head: 160,931
  Seg Head: 206,342
  YOLO Head: 1,462,791
🔄 1. Starte SWA: Mittle 6 Checkpoints...
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v4_0_best.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_56672.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_55440.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_54208.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_52976.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v4_0_step_51744.pth
✅ Mittelung abgeschlossen.
🏗️ 2. Lade SWA-Gewichte in Trainings-Struktur...
🧬 3. Rufe switch_to_deploy() auf (RepConv Faltung)...
✨ 12 RepConv-Layer erfolgreich gefaltet!
🧬 3. Transformiere Stereo-Gewichte für Hailo (Grouped Convs)...
🔄 4. Schalte gesamtes Modell live in den Deploy-Modus...
🟢 Alle Module erfolgreich auf Deploy-Modus umgeschaltet!
💾 Finales Deploy-Modell gespeichert unter: 

/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:663: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:1186: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../to

🎉 Export abgeschlossen! Die Datei 'hexapod_v4_0_deploy.onnx' ist jetzt zu 100% Hailo-Ready.


In [5]:
#!pip install onnxsim
!python -m onnxsim hexapod_v4_0_deploy.onnx ./onnx_split/hexapod_v4_0_simplified.onnx --no-large-tensor

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                 ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Abs             │ 1              │ 1                │
│ Add             │ 23             │ 23               │
│ Cast            │ 23             │ 0                │
│ Clip            │ 50             │ 50               │
│ Concat          │ 64             │ 15               │
│ Constant        │ 602            │ 272              │
│ ConstantOfShape │ 23             │ 0                │
│ Conv            │ 127            │ 127              │
│ Div             │ 6              │ 6                │
│ Exp             │ 1              │ 1                │
│ Identity        │ 46             │ 0                │
│ MaxPool         │ 3              │ 3                │
│ Mul             │ 8              │ 8                │
│ Pad             │ 23             │ 23               │
│

In [6]:
"""
export_split_cells.py — Zum Einfügen als Zellen in dein bestehendes Jupyter Notebook.
Läuft DIREKT nach dem SWA + Fold + Deploy Block weiter.

Voraussetzung: `model` ist im Speicher, deploy-gefaltet und auf DEVICE.

Einfach als neue Zelle(n) nach dem ONNX-Export-Block einfügen.
"""

# =====================================================================
# ZELLE 1: Wrapper-Klassen definieren
# =====================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import subprocess

SPLIT_DIR = "onnx_split"
os.makedirs(SPLIT_DIR, exist_ok=True)

class BackboneWrapper(nn.Module):
    """Input: gray_img [1,1,480,640] → f_s4, f_s8, f_s16, f_s32"""
    def __init__(self, full_model):
        super().__init__()
        self.backbone = full_model.backbone
    def forward(self, gray_img):
        f = self.backbone(gray_img)
        return f[0], f[1], f[2], f[3]

class GeometryWrapper(nn.Module):
    """
    Inputs: f_s4_l, f_s8_l, f_s4_r, f_s8_r, img_l
    Outputs: disp_final, normals_s4, disp_s8
    """
    def __init__(self, full_model):
        super().__init__()
        self.geo_stem = full_model.geo_stem
        self.normals_head = full_model.normals_head
        self.stereo_head = full_model.stereo_head
        
    def forward(self, f_s4_l, f_s8_l, f_s4_r, f_s8_r, img_l):
        geo_feat_l = self.geo_stem(f_s4_l)
        geo_feat_r = self.geo_stem(f_s4_r)
        
        # 🚨 FIX: Erwartet nur noch 1 Wert (s4), da s1 entfernt wurde
        normals_s4 = self.normals_head(
            f_s4_l, f_s8_l, img_l, geo_features=geo_feat_l
        )
        
        final_disp, disp_s8 = self.stereo_head(
            f_s8_l, f_s8_r, f_s4_l, img_l,
            normals_s4=normals_s4,
            geo_features_l=geo_feat_l,
            geo_features_r=geo_feat_r
        )
        # normals_s1 aus dem Return entfernt
        return final_disp, normals_s4, disp_s8

class DetectionWrapper(nn.Module):
    """
    Inputs: f_s4_l, f_s8_l, f_s16_l, f_s32_l, normals_s4
    Outputs: seg, yolo_s8, yolo_s16, yolo_s32
    """
    def __init__(self, full_model):
        super().__init__()
        self.sppf = full_model.sppf
        self.fpn_neck = full_model.fpn_neck
        # 🚨 FIX: CBAM Attribute entfernt!
        self.seg_head = full_model.seg_head
        self.yolo_head = full_model.yolo_head
        
    def forward(self, f_s4_l, f_s8_l, f_s16_l, f_s32_l, normals_s4):
        seg = self.seg_head(f_s4_l, f_s16_l, normals_s4=normals_s4)
        
        f_s32_sppf = self.sppf(f_s32_l)
        fpn_s8, fpn_s16, fpn_s32 = self.fpn_neck(f_s8_l, f_s16_l, f_s32_sppf)
        
        # 🚨 FIX: FPN Features gehen ohne CBAM direkt in den YOLO Head
        yolo_s8, yolo_s16, yolo_s32 = self.yolo_head(fpn_s8, fpn_s16, fpn_s32)
        
        return seg, yolo_s8, yolo_s16, yolo_s32


# =====================================================================
# ZELLE 2: Export-Hilfsfunktion
# =====================================================================

def export_onnx(module, dummy_inputs, input_names, output_names, filename):
    """Exportiert als ONNX und vereinfacht mit onnxsim."""
    raw_path = os.path.join(SPLIT_DIR, filename)
    sim_path = os.path.join(SPLIT_DIR, filename.replace('.onnx', '_simplified.onnx'))
    
    print(f"\n🔄 Exportiere {filename}...")
    torch.onnx.export(
        module, dummy_inputs, raw_path,
        input_names=input_names,
        output_names=output_names,
        opset_version=13,
        do_constant_folding=True,
    )
    print(f"   ✅ {raw_path}")
    
    try:
        subprocess.run(['python', '-m', 'onnxsim', raw_path, sim_path],
                       check=True, capture_output=True, text=True)
        print(f"   ✅ Vereinfacht: {sim_path}")
        return sim_path
    except Exception as e:
        print(f"   ⚠️  onnxsim fehlgeschlagen ({e}), verwende Original")
        return raw_path


# =====================================================================
# ZELLE 3: Alle 3 Sub-Modelle exportieren
# =====================================================================

print("=" * 60)
print("🔪 SPLIT-EXPORT: 3 Sub-Modelle aus geladenem Deploy-Modell")
print("=" * 60)

model.eval()

# 🚨 NEU: Kanäle dynamisch direkt aus dem Backbone auslesen!
ch_s4, ch_s8, ch_s16, ch_s32 = model.backbone.feature_info.channels()
print(f"Dynamische Kanal-Tiefen für Export: s4={ch_s4}, s8={ch_s8}, s16={ch_s16}, s32={ch_s32}")

# --- A: Backbone ---
bb = BackboneWrapper(model).eval().to(DEVICE)
export_onnx(
    bb,
    (torch.randn(1, 1, 480, 640, device=DEVICE),),
    ['gray_img'],
    ['f_s4', 'f_s8', 'f_s16', 'f_s32'],
    'hexapod_backbone.onnx',
)
del bb

# --- B: Geometry ---
geo = GeometryWrapper(model).eval().to(DEVICE)
export_onnx(
    geo,
    (
        torch.randn(1, ch_s4, 120, 160, device=DEVICE),   # f_s4_l
        torch.randn(1, ch_s8, 60, 80, device=DEVICE),     # f_s8_l
        torch.randn(1, ch_s4, 120, 160, device=DEVICE),   # f_s4_r
        torch.randn(1, ch_s8, 60, 80, device=DEVICE),     # f_s8_r
        torch.randn(1, 1, 480, 640, device=DEVICE),       # img_l
    ),
    ['f_s4_l', 'f_s8_l', 'f_s4_r', 'f_s8_r', 'img_l'],
    # 🚨 FIX: normals_s1 aus den output_names entfernt!
    ['disp_final', 'normals_s4', 'disp_s8'],
    'hexapod_geometry.onnx',
)
del geo

# --- C: Detection ---
det = DetectionWrapper(model).eval().to(DEVICE)
export_onnx(
    det,
    (
        torch.randn(1, ch_s4, 120, 160, device=DEVICE),   # f_s4_l
        torch.randn(1, ch_s8, 60, 80, device=DEVICE),     # f_s8_l
        torch.randn(1, ch_s16, 30, 40, device=DEVICE),    # f_s16_l
        torch.randn(1, ch_s32, 15, 20, device=DEVICE),    # f_s32_l
        torch.randn(1, 3, 120, 160, device=DEVICE),       # normals_s4
    ),
    ['f_s4_l', 'f_s8_l', 'f_s16_l', 'f_s32_l', 'normals_s4'],
    ['seg', 'yolo_s8', 'yolo_s16', 'yolo_s32'],
    'hexapod_detection.onnx',
)
del det

# --- Zusammenfassung ---
print("\n" + "=" * 60)
print("🎉 SPLIT-EXPORT ABGESCHLOSSEN!")
print("=" * 60)
for f in ['hexapod_backbone', 'hexapod_geometry', 'hexapod_detection']:
    sim = os.path.join(SPLIT_DIR, f'{f}_simplified.onnx')
    raw = os.path.join(SPLIT_DIR, f'{f}.onnx')
    path = sim if os.path.exists(sim) else raw
    size_mb = os.path.getsize(path) / 1e6
    print(f"   {f:25s} → {size_mb:.1f} MB")

print(f"\nNächster Schritt: python quantizer_split.py")
print(f"   (oder einzeln: python quantizer_split.py backbone)")


🔪 SPLIT-EXPORT: 3 Sub-Modelle aus geladenem Deploy-Modell
Dynamische Kanal-Tiefen für Export: s4=32, s8=48, s16=136, s32=448

🔄 Exportiere hexapod_backbone.onnx...
   ✅ onnx_split/hexapod_backbone.onnx
   ✅ Vereinfacht: onnx_split/hexapod_backbone_simplified.onnx

🔄 Exportiere hexapod_geometry.onnx...


/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:663: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:1186: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../to

   ✅ onnx_split/hexapod_geometry.onnx
   ✅ Vereinfacht: onnx_split/hexapod_geometry_simplified.onnx

🔄 Exportiere hexapod_detection.onnx...
   ✅ onnx_split/hexapod_detection.onnx
   ✅ Vereinfacht: onnx_split/hexapod_detection_simplified.onnx

🎉 SPLIT-EXPORT ABGESCHLOSSEN!
   hexapod_backbone          → 14.0 MB
   hexapod_geometry          → 1.1 MB
   hexapod_detection         → 9.1 MB

Nächster Schritt: python quantizer_split.py
   (oder einzeln: python quantizer_split.py backbone)


In [21]:
"""
export_combined_cells.py — Exportiert Geometry+Detection als ein kombiniertes ONNX.
Zum Einfügen als Zelle in dein Jupyter Notebook NACH dem SWA/Fold/Deploy Block.

Voraussetzung: `model` existiert im Speicher, deploy-ready, auf DEVICE.

Das Ziel ist ein 2-HEF-Setup:
  HEF A: Backbone (schon kompiliert!)
  HEF B: Geometry + Detection (alles andere)

Vorteile gegenüber 3 HEFs:
  - normals_s4 bleibt intern (kein PCIe-Transfer)
  - Ein Context-Switch weniger
  - Einfachere Runtime-Pipeline
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import subprocess

SPLIT_DIR = "onnx_split"
os.makedirs(SPLIT_DIR, exist_ok=True)


class GeometryDetectionWrapper(nn.Module):
    """
    Combined Geometry and Detection Wrapper
    """
    def __init__(self, full_model):
        super().__init__()
        # --- Geometry Heads ---
        self.geo_stem = full_model.geo_stem
        self.normals_head = full_model.normals_head
        self.stereo_head = full_model.stereo_head
        
        # --- Detection Heads ---
        self.sppf = full_model.sppf
        self.fpn_neck = full_model.fpn_neck
        # 🚨 FIX: CBAM (self.cbam_s8 etc.) wurde hier komplett entfernt!
        self.seg_head = full_model.seg_head
        self.yolo_head = full_model.yolo_head
        
    def forward(self, f_s4_l, f_s8_l, f_s4_r, f_s8_r, f_s16_l, f_s32_l, img_l):
        # ==========================================
        # 1. GEOMETRY PASS
        # ==========================================
        geo_feat_l = self.geo_stem(f_s4_l)
        geo_feat_r = self.geo_stem(f_s4_r)
        
        # (Beachte: normals_s1 wurde wie besprochen entfernt)
        normals_s4 = self.normals_head(
            f_s4_l, f_s8_l, img_l, geo_features=geo_feat_l
        )
        
        final_disp, disp_s8 = self.stereo_head(
            f_s8_l, f_s8_r, f_s4_l, img_l,
            normals_s4=normals_s4,
            geo_features_l=geo_feat_l,
            geo_features_r=geo_feat_r
        )

        # ==========================================
        # 2. DETECTION PASS
        # ==========================================
        seg = self.seg_head(f_s4_l, f_s16_l, normals_s4=normals_s4)
        
        f_s32_sppf = self.sppf(f_s32_l)
        fpn_s8, fpn_s16, fpn_s32 = self.fpn_neck(f_s8_l, f_s16_l, f_s32_sppf)
        
        # 🚨 FIX: FPN Features fließen ohne CBAM-Attention direkt in YOLO
        yolo_s8, yolo_s16, yolo_s32 = self.yolo_head(fpn_s8, fpn_s16, fpn_s32)
        
        return final_disp, normals_s4, disp_s8, seg, yolo_s8, yolo_s16, yolo_s32


# =====================================================================
# EXPORT
# =====================================================================
def export_onnx(module, dummy_inputs, input_names, output_names, filename):
    """Exportiert als ONNX und vereinfacht mit onnxsim."""
    raw_path = os.path.join(SPLIT_DIR, filename)
    sim_path = os.path.join(SPLIT_DIR, filename.replace('.onnx', '_simplified.onnx'))
    
    print(f"\n🔄 Exportiere {filename}...")
    torch.onnx.export(
        module, dummy_inputs, raw_path,
        input_names=input_names,
        output_names=output_names,
        opset_version=13,
        do_constant_folding=True,
    )
    print(f"   ✅ {raw_path}")
    
    try:
        subprocess.run(['python', '-m', 'onnxsim', raw_path, sim_path],
                       check=True, capture_output=True, text=True)
        print(f"   ✅ Vereinfacht: {sim_path}")
    except Exception as e:
        print(f"   ⚠️  onnxsim fehlgeschlagen ({e}), verwende Original")


# ==========================================
# COMBINED EXPORT
# ==========================================

# 🚨 Sicherstellen, dass wir die dynamischen Kanäle aus dem Backbone haben
ch_s4, ch_s8, ch_s16, ch_s32 = model.backbone.feature_info.channels()
print(f"Dynamische Kanal-Tiefen für Export: s4={ch_s4}, s8={ch_s8}, s16={ch_s16}, s32={ch_s32}")

model.eval()
combined = GeometryDetectionWrapper(model).eval().to(DEVICE)

export_onnx(
    combined,
    (
        # 🚨 FIX: Hier nutzen wir jetzt ch_s4, ch_s8, etc. statt 24, 40...
        torch.randn(1, ch_s4, 120, 160, device=DEVICE),   # f_s4_l
        torch.randn(1, ch_s8, 60, 80, device=DEVICE),     # f_s8_l
        torch.randn(1, ch_s4, 120, 160, device=DEVICE),   # f_s4_r
        torch.randn(1, ch_s8, 60, 80, device=DEVICE),     # f_s8_r
        torch.randn(1, ch_s16, 30, 40, device=DEVICE),    # f_s16_l
        torch.randn(1, ch_s32, 15, 20, device=DEVICE),    # f_s32_l
        torch.randn(1, 1, 480, 640, device=DEVICE),       # img_l
    ),
    input_names=['f_s4_l', 'f_s8_l', 'f_s4_r', 'f_s8_r', 'f_s16_l', 'f_s32_l', 'img_l'],
    # (Beachte: normals_s1 ist hier wieder aus den output_names entfernt, wie vorhin besprochen)
    output_names=['disp_final', 'normals_s4', 'disp_s8', 'seg', 'yolo_s8', 'yolo_s16', 'yolo_s32'],
    filename='hexapod_geo_det_combined.onnx',
)

del combined

# Größe anzeigen
for f in ['hexapod_geo_det_combined', 'hexapod_geo_det_combined_simplified']:
    path = os.path.join(SPLIT_DIR, f'{f}.onnx')
    if os.path.exists(path):
        print(f"   {f}: {os.path.getsize(path)/1e6:.1f} MB")

print(f"\n📋 Nächste Schritte:")
print(f"   1. Füge 'combined' als Option in quantizer_split.py ein")
print(f"   2. python quantizer_split.py combined")
print(f"   3. Falls Compile klappt: 2-HEF Pipeline testen!")


Dynamische Kanal-Tiefen für Export: s4=32, s8=48, s16=136, s32=448

🔄 Exportiere hexapod_geo_det_combined.onnx...


/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:663: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:1186: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../to

   ✅ onnx_split/hexapod_geo_det_combined.onnx
   ✅ Vereinfacht: onnx_split/hexapod_geo_det_combined_simplified.onnx
   hexapod_geo_det_combined: 10.3 MB
   hexapod_geo_det_combined_simplified: 10.2 MB

📋 Nächste Schritte:
   1. Füge 'combined' als Option in quantizer_split.py ein
   2. python quantizer_split.py combined
   3. Falls Compile klappt: 2-HEF Pipeline testen!


In [1]:
import onnx

# Lade das reine, exportierte ONNX-Modell (oder das _simplified)
onnx_path = "onnx_split/hexapod_backbone.onnx"
print(f"Lese Graphen von: {onnx_path}\n")

model_onnx = onnx.load(onnx_path)

print("=== INPUTS ===")
for inp in model_onnx.graph.input:
    # Versuche die Shape auszulesen (Manche Dimensionen können dynamisch '0' sein)
    try:
        shape = [dim.dim_value for dim in inp.type.tensor_type.shape.dim]
    except:
        shape = "Unknown"
    print(f"Name: {inp.name:15s} | Shape: {shape}")

print("\n=== OUTPUTS ===")
for out in model_onnx.graph.output:
    try:
        shape = [dim.dim_value for dim in out.type.tensor_type.shape.dim]
    except:
        shape = "Unknown"
    print(f"Name: {out.name:15s} | Shape: {shape}")

Lese Graphen von: onnx_split/hexapod_backbone.onnx

=== INPUTS ===
Name: gray_img        | Shape: [1, 1, 480, 640]

=== OUTPUTS ===
Name: f_s4            | Shape: [1, 32, 120, 160]
Name: f_s8            | Shape: [1, 48, 60, 80]
Name: f_s16           | Shape: [1, 136, 30, 40]
Name: f_s32           | Shape: [1, 448, 15, 20]


In [4]:
import onnx

model_onnx = onnx.load("hexapod_v4_0_deploy.onnx")
onnx.checker.check_model(model_onnx)
print("✅ ONNX-Graph ist valide")
print(f"Opset: {model_onnx.opset_import[0].version}")

# Inputs/Outputs anzeigen
for inp in model_onnx.graph.input:
    print(f"Input:  {inp.name} — {[d.dim_value for d in inp.type.tensor_type.shape.dim]}")
for out in model_onnx.graph.output:
    print(f"Output: {out.name} — {[d.dim_value for d in out.type.tensor_type.shape.dim]}")



✅ ONNX-Graph ist valide
Opset: 13
Input:  input_layer1 — [1, 1, 480, 640]
Input:  input_layer2 — [1, 1, 480, 640]
Output: disp_final — [1, 1, 120, 160]
Output: seg — [1, 6, 120, 160]
Output: disp_s8 — [1, 1, 60, 80]
Output: normals_s1 — [1, 3, 120, 160]
Output: yolo_s8 — [1, 45, 60, 80]
Output: yolo_s16 — [1, 45, 30, 40]
Output: yolo_s32 — [1, 45, 15, 20]


In [5]:
import onnxruntime as ort
import numpy as np
import copy
import torch.nn.functional as F

print("\n" + "="*50)
print("🔬 DER ULTIMATIVE 3-WEGE-VERGLEICH & EXPORT")
print("="*50)

# ====================================================================
# ZUSTAND 1: DAS ORIGINAL (Training-Struktur, nackte SWA-Gewichte)
# ====================================================================
print("🔄 1. Baue isoliertes Original-Modell auf...")
config_orig = copy.deepcopy(CONFIG)
config_orig['deploy'] = False  # Zwingend False für SWA!

model_orig = FusedHexapodModel(config_orig)

# Hole die rohen SWA-Gewichte (Du nutzt deine create_swa_and_fold Logik,
# aber wir machen hier nur den SWA-Teil, OHNE zu falten!)
# ANNAHME: Du hast 'raw_swa_state_dict' vorher im Skript erzeugt.
model_orig.load_state_dict(raw_swa_state_dict, strict=False)
model_orig.eval().cpu()

# ====================================================================
# ZUSTAND 2: PYTORCH DEPLOY (Live-Umbau per Deepcopy)
# ====================================================================
print("🏗️ 2. Erstelle Deploy-Modell per Deepcopy...")
model_deploy = copy.deepcopy(model_orig)

print("🧬 3. Falte RepConvs & Transformiere Stereo...")
for m in model_deploy.modules():
    if hasattr(m, 'switch_to_deploy'):
        m.switch_to_deploy()

prep_stereo_for_deploy(model_deploy.stereo_head)

print("🔄 4. Lege alle Deploy-Flags live um...")
def enable_deploy_mode_recursively(net):
    if hasattr(net, 'deploy_mode'): net.deploy_mode = True
    for m in net.modules():
        if hasattr(m, 'deploy'): m.deploy = True

enable_deploy_mode_recursively(model_deploy)
model_deploy.eval()

# Speichere das saubere Modell
output_path_deploy = "checkpoints/checkpoint_v4_0_deploy.pth"
os.makedirs(os.path.dirname(output_path_deploy), exist_ok=True)
torch.save(model_deploy.state_dict(), output_path_deploy)

# ====================================================================
# ZUSTAND 3: ONNX EXPORT & INFERENZ
# ====================================================================
print("\n🚀 5. Starte ONNX Export...")
#dummy_l = (torch.randn(1, 1, 480, 640) * 10.0 + 2.0)
#dummy_r = (torch.randn(1, 1, 480, 640) * 10.0 + 2.0)
dummy_l = (torch.randn(1, 1, 480, 640))
dummy_r = (torch.randn(1, 1, 480, 640))
           
onnx_path = "hexapod_v4_0_deploy.onnx"
torch.onnx.export(
    model_deploy, (dummy_l, dummy_r), onnx_path,
    input_names=['input_layer1', 'input_layer2'],
    output_names=['disp_final', 'seg', 'disp_s8', 'normals_s1', 'yolo_s8', 'yolo_s16', 'yolo_s32'],
    opset_version=13, do_constant_folding=True
)

sess = ort.InferenceSession(onnx_path)
out_onnx = sess.run(None, {'input_layer1': dummy_l.numpy(), 'input_layer2': dummy_r.numpy()})

# ====================================================================
# INFERENZ-DURCHLÄUFE FÜR TEST
# ====================================================================
with torch.no_grad():
    out_orig = model_orig(dummy_l, dummy_r, use_normals_for_stereo=True) # Jetzt 7 Elemente
    out_deploy = model_deploy(dummy_l, dummy_r) # Jetzt 7 Elemente




names = ['disp_final', 'seg', 'disp_s8', 'normals_s1', 'yolo_s8', 'yolo_s16', 'yolo_s32']

print("\n🔍 TEST A: PyTorch Original vs. PyTorch Deploy")
for i, name in enumerate(names):
    t_orig = out_orig[i].cpu().numpy()
    t_deploy = out_deploy[i].cpu().numpy()

    diff = np.abs(t_orig - t_deploy).max()
    status = "✅ OK" if diff < 1e-3 else ("⚠️ GRENZWERTIG" if diff < 1e-2 else "❌ FEHLER")
    print(f"  {name:12s}: max_diff = {diff:.8f} {status};{t_orig.max()}-{t_deploy.max()}")

print("\n🔍 TEST B: PyTorch Deploy vs. ONNX")
for i, name in enumerate(names):
    t_deploy = out_deploy[i].numpy()
    t_onnx = out_onnx[i]
    
    diff = np.abs(t_deploy - t_onnx).max()
    status = "✅ OK" if diff < 1e-4 else ("⚠️ (ONNX-Rundung)" if diff < 1e-2 else "❌ FEHLER")
    print(f"  {name:12s}: max_diff = {diff:.8f} {status};{t_deploy.max()}-{t_onnx.max()}")


🔬 DER ULTIMATIVE 3-WEGE-VERGLEICH & EXPORT
🔄 1. Baue isoliertes Original-Modell auf...


NameError: name 'CONFIG' is not defined

In [56]:
# Modell 1: Original (Training-Modus, keine Grouped Convs)
deploy_config_orig = copy.deepcopy(CONFIG)
deploy_config_orig['deploy'] = False
model_orig = FusedHexapodModel(deploy_config_orig)
model_orig, _ = create_swa_and_fold(model_orig, checkpoint_paths)
model_orig.eval()

# Modell 2: Deploy (mit prep_stereo_for_deploy + enable_deploy_mode)
deploy_config_deploy = copy.deepcopy(CONFIG)
deploy_config_deploy['deploy'] = False  # Erst False für SWA!
model_deploy = FusedHexapodModel(deploy_config_deploy)
model_deploy, _ = create_swa_and_fold(model_deploy, checkpoint_paths)
prep_stereo_for_deploy(model_deploy.stereo_head)
enable_deploy_mode_recursively(model_deploy)
model_deploy.eval();

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
/tmp/ipykernel_3383904/210025356.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file

  ✅ Backbone first conv: 3→1 channel (luminance merge)
  ✅ SPPF Module injected at s32 (960 channels)
  ✅ CBAM Attention Modules injected after FPN
  ✅ Geometry Stem Modules injected after Backbone
  Total parameters: 7,237,261
  Normals Head: 158,790
  Seg Head: 36,950
  YOLO Head: 1,462,791
🔄 1. Starte SWA: Mittle 6 Checkpoints...
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_best.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_59136.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_57904.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_56672.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_55440.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_54208.pth
✅ Mittelung abgeschlossen.
🏗️ 2. Lade SWA-Gewichte in Trainings-Struktur...
🧬 3. Rufe switch_to_deploy() auf (RepConv Faltung)...


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


✨ 12 RepConv-Layer erfolgreich gefaltet!
  ✅ Backbone first conv: 3→1 channel (luminance merge)
  ✅ SPPF Module injected at s32 (960 channels)
  ✅ CBAM Attention Modules injected after FPN
  ✅ Geometry Stem Modules injected after Backbone
  Total parameters: 7,237,261
  Normals Head: 158,790
  Seg Head: 36,950
  YOLO Head: 1,462,791
🔄 1. Starte SWA: Mittle 6 Checkpoints...
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_best.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_59136.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_57904.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_56672.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_55440.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_54208.pth
✅ Mittelung abgeschlossen.
🏗️ 2. Lade SWA-Gewichte in Trainings-Struktur...
🧬 3. Rufe switch_to_deploy() auf (RepConv Faltung)...
✨ 12 RepConv-Layer erfolgreich gefaltet!


In [18]:
with torch.no_grad():
    out_orig = model_orig(dummy_l, dummy_r)
    # Original: [final_disp, seg, det, disp_s8, normals_s1]
    print(type(out_orig[2]))
    print(len(out_orig[2]) if hasattr(out_orig[2], '__len__') else "kein len")
    print(out_orig[2])
    mapped_orig = [
        out_orig[0],     # disp_final
        out_orig[1],     # seg
        out_orig[3],     # disp_s8
        out_orig[4],     # normals_s1
        out_orig[2][0],  # yolo_s8
        out_orig[2][1],  # yolo_s16
        out_orig[2][2],  # yolo_s32
    ]
    
    out_deploy = model_deploy(dummy_l, dummy_r)
    # Deploy: [final_disp, seg, disp_s8, normals_s1, yolo_s8, yolo_s16, yolo_s32]
    mapped_deploy = list(out_deploy)

<class 'torch.Tensor'>
1
tensor([[[[13.4951, 13.9991, 14.4898,  ...,  4.2975,  6.1391,  6.0730],
          [13.4990, 13.9987, 14.4956,  ...,  4.1497,  5.1397,  5.6227],
          [13.3778, 13.9829, 14.4922,  ...,  5.3935, 13.1903,  5.2497],
          ...,
          [13.4990,  5.0081,  1.4991,  ...,  6.4597, 13.2572,  6.8135],
          [13.4834,  0.6529,  1.0043,  ...,  3.6770, 10.7839, 13.6024],
          [13.3604,  6.5400,  4.8317,  ...,  5.4919,  9.0213,  6.5857]]]])


IndexError: index 1 is out of bounds for dimension 0 with size 1

In [19]:
print("deploy_mode:", model_orig.deploy_mode)
print("out_orig length:", len(out_orig))
for i, o in enumerate(out_orig):
    if isinstance(o, torch.Tensor):
        print(f"  [{i}]: Tensor {o.shape}")
    elif isinstance(o, list):
        print(f"  [{i}]: Liste mit {len(o)} Elementen")
        for j, oo in enumerate(o):
            print(f"    [{j}]: {oo.shape}")

deploy_mode: False
out_orig length: 7
  [0]: Tensor torch.Size([1, 1, 480, 640])
  [1]: Tensor torch.Size([1, 6, 120, 160])
  [2]: Tensor torch.Size([1, 1, 60, 80])
  [3]: Tensor torch.Size([1, 3, 480, 640])
  [4]: Tensor torch.Size([1, 45, 60, 80])
  [5]: Tensor torch.Size([1, 45, 30, 40])
  [6]: Tensor torch.Size([1, 45, 15, 20])


In [68]:
with torch.no_grad():
    out_orig = model_orig(dummy_l, dummy_r, use_normals_for_stereo=True)
    out_deploy = model_deploy(dummy_l, dummy_r)

names = ['disp_final', 'seg', 'disp_s8', 'normals_s1', 'yolo_s8', 'yolo_s16', 'yolo_s32']

print("\n🔍 TEST A: PyTorch Original vs. PyTorch Deploy")
for i, name in enumerate(names):
    t_orig = out_orig[i].cpu().numpy()
    t_deploy = out_deploy[i].cpu().numpy()
    diff = np.abs(t_orig - t_deploy).max()
    status = "✅ OK" if diff < 1e-3 else ("⚠️ GRENZWERTIG" if diff < 1e-2 else "❌ FEHLER")
    print(f"  {name:12s}: max_diff = {diff:.8f} {status}")

Training channel:
coarse_normals_s4 deploy: 0.999980628490448
normals_coarse deploy: 0.9987637400627136
refined deploy: 0.871442973613739
normals_s1 vor norm: 0.9999988675117493
training
vol_s8 shape: torch.Size([1, 24, 60, 80])
vol_ctx shape: torch.Size([1, 24, 60, 80])
vol_s8 max: 14.721708297729492
vol_ctx max: 71.479248046875
prob train: 1.0
disp_s8 train: 22.401683807373047
disp_s4 max: 11.275081634521484
final_disp max: 40.711143493652344
training: normals_s4_for_stereo is None: False
Deployment channel:
coarse_normals_s4 deploy: 0.999980628490448
normals_coarse deploy: 0.9987637400627136
refined deploy: 0.8714431524276733
normals_s1 vor norm: 1.6401277780532837
deploy
vol_s8 shape: torch.Size([1, 24, 60, 80])
vol_ctx shape: torch.Size([1, 24, 60, 80])
vol_s8 max: 14.721707344055176
vol_ctx max: 71.47925567626953
prob deploy: 0.9999990463256836
disp_s8 deploy: 22.401676177978516
disp_s4 max: 11.275090217590332
final_disp max: 40.71117401123047
deploy: normals_s4_for_stereo is Non

In [21]:
# Gleiche Inputs für beide NormalsHeads
torch.manual_seed(0)
dummy_feat_s4 = torch.randn(1, 24, 120, 160)
dummy_feat_s8 = torch.randn(1, 40, 60, 80)
dummy_gray    = torch.randn(1, 1, 480, 640)
dummy_geo     = torch.randn(1, 32, 120, 160)

with torch.no_grad():
    n_orig   = model_orig.normals_head(dummy_feat_s4, dummy_feat_s8, dummy_gray, dummy_geo)
    n_deploy = model_deploy.normals_head(dummy_feat_s4, dummy_feat_s8, dummy_gray, dummy_geo)

print("normals_s1 diff:", (n_orig[0] - n_deploy[0]).abs().max().item())
print("normals_s4 diff:", (n_orig[1] - n_deploy[1]).abs().max().item())

normals_s1 diff: 1.116782784461975
normals_s4 diff: 1.7881393432617188e-07


In [22]:
torch.manual_seed(0)
x = torch.randn(1, 3, 480, 640)

# Training
out_train = F.normalize(x, p=2, dim=1, eps=1e-4)

# Deploy
v_max, _ = torch.max(torch.abs(x), dim=1, keepdim=True)
s = x / (v_max + 1e-6)
l2_sq = torch.sum(s * s, dim=1, keepdim=True)
l2_norm = torch.sqrt(l2_sq)
divisor = torch.clamp(l2_norm * (v_max + 1e-6), min=1e-4)
out_deploy = x / divisor

print("diff:", (out_train - out_deploy).abs().max().item())

diff: 2.384185791015625e-07


In [4]:
!pip install onnxsim
!python -m onnxsim hexapod_v4_0_deploy.onnx hexapod_v4_0_simplified.onnx --no-large-tensor

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                   ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Abs               │ 5              │ 3                │
│ Add               │ 31             │ 30               │
│ AveragePool       │ 1              │ 1                │
│ Cast              │ 23             │ 0                │
│ Clip              │ 2              │ 2                │
│ Concat            │ 70             │ 20               │
│ Constant          │ 558            │ 319              │
│ ConstantOfShape   │ 23             │ 0                │
│ Conv              │ 171            │ 169              │
│ Div               │ 7              │ 7                │
│ Exp               │ 1              │ 1                │
│ GlobalAveragePool │ 3              │ 3                │
│ HardSigmoid       │ 32             │ 32               │
│ Identity          │ 48 

In [13]:
import onnx
model = onnx.load("hexapod_v4_0_simplified.onnx")

# Nur Graph-Struktur ohne Gewichte
with open("model_structure.txt", "w") as f:
    for node in model.graph.node:
        f.write(f"{node.op_type:20s} | {node.name:50s} | "
                f"in={list(node.input)} out={list(node.output)}\n")
    f.write("\n--- INPUTS ---\n")
    for inp in model.graph.input:
        shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
        f.write(f"{inp.name}: {shape}\n")
    f.write("\n--- OUTPUTS ---\n")
    for out in model.graph.output:
        shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
        f.write(f"{out.name}: {shape}\n")

print("✅ model_structure.txt geschrieben")

✅ model_structure.txt geschrieben


In [14]:
import torch

# 1. Modul initialisieren (mit deinen Parametern)
max_disp = 24
in_channels = 32
model = CoarseCostVolume(max_disp=max_disp, in_channels=in_channels)
model.eval() # Sehr wichtig: Dropout & BatchNorm einfrieren!

# 2. Realistische Dummy-Daten erzeugen (Batch=1 für Deployment-Test)
B, C, H, W = 1, 32, 60, 80
feat_l = torch.randn(B, C, H, W)
feat_r = torch.randn(B, C, H, W)

print("🧪 Starte mathematischen Äquivalenz-Test...")

with torch.no_grad():
    # --- DURCHLAUF 1: Alter Trainings-Code (GPU-optimiert) ---
    model.deploy = False
    out_training = model(feat_l, feat_r)
    print(f"✅ Training-Output Shape: {out_training.shape}")

    # --- DURCHLAUF 2: Neuer NPU-Code (Hailo-optimiert) ---
    model.deploy = True
    out_deploy = model(feat_l, feat_r)
    print(f"✅ Deploy-Output Shape:   {out_deploy.shape}")

# 3. Numerischer Vergleich
# Wir berechnen den maximalen Unterschied zwischen beiden Tensoren
max_diff = torch.abs(out_training - out_deploy).max().item()

print("-" * 40)
print(f"📊 Maximale Abweichung: {max_diff:.10f}")

# Eine Abweichung von < 1e-6 (0.000001) gilt bei Float32 als "perfekt identisch"
if max_diff < 1e-6:
    print("🎉 ERFOLG: Die Pfade sind mathematisch zu 100% identisch!")
else:
    print("⚠️ FEHLER: Es gibt eine Abweichung.")

🧪 Starte mathematischen Äquivalenz-Test...
✅ Training-Output Shape: torch.Size([1, 24, 60, 80])


AttributeError: 'CoarseCostVolume' object has no attribute 'corr_grouped'

In [19]:
model_onnx = onnx.load("hexapod_v4_0_simplified.onnx")
onnx.checker.check_model(model_onnx)
for out in model_onnx.graph.output:
    print(f"Output: {out.name} — {[d.dim_value for d in out.type.tensor_type.shape.dim]}")

Output: disp_final — [1, 1, 480, 640]
Output: seg — [1, 6, 120, 160]
Output: disp_s8 — [1, 1, 60, 80]
Output: normals — [1, 3, 480, 640]
Output: yolo_s8 — [1, 45, 60, 80]
Output: yolo_s16 — [1, 45, 30, 40]
Output: yolo_s32 — [1, 45, 15, 20]


In [15]:
import numpy as np
import cv2, glob, random, os

random.seed(42)

TARTAN_TRAIN_ROOT = "/home/slarc/datasets/TartanAir"
N_SAMPLES = 1024

# Alle TartanAir Bildpaare sammeln
samples = []
for env in sorted(glob.glob(os.path.join(TARTAN_TRAIN_ROOT, '*'))):
    for diff in ['Easy', 'Hard']:
        for traj in sorted(glob.glob(os.path.join(env, diff, 'P*'))):
            left_dir = os.path.join(traj, 'image_left')
            right_dir = os.path.join(traj, 'image_right')
            if not all(os.path.exists(d) for d in [left_dir, right_dir]):
                continue
            for lp in sorted(glob.glob(os.path.join(left_dir, '*.png'))):
                fn = os.path.basename(lp).replace('_left.png', '')
                rp = os.path.join(right_dir, fn + '_right.png')
                if os.path.exists(rp):
                    samples.append({'l': lp, 'r': rp})

print(f"✅ {len(samples)} Bildpaare gefunden")
pairs = random.sample(samples, min(N_SAMPLES, len(samples)))

calib_left, calib_right = [], []
for s in pairs:
    for path, out in [(s['l'], calib_left), (s['r'], calib_right)]:
        img = cv2.imread(path)
        img = cv2.resize(img, (640, 480))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
        # Keine Normalisierung! Das übernimmt jetzt das .alls
        out.append(gray[np.newaxis])  # [1, H, W], float32, 0-255

np.save('calib_left.npy', np.array(calib_left))
np.save('calib_right.npy', np.array(calib_right))
print(f"✅ calib_left.npy / calib_right.npy gespeichert — {len(pairs)} Paare, range [0, 255]")

✅ 84824 Bildpaare gefunden
✅ calib_left.npy / calib_right.npy gespeichert — 1024 Paare, range [0, 255]
